# Classifier Results - Quick Overview
This notebook loads and displays the final classification results with statistical tests.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import permutation_test
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
print(f"Random seed set to {SEED}")

Random seed set to 42


In [2]:
def paired_permutation_tests(df, model_a, model_b):
    """
    Performs paired permutation tests between two models across all tasks and orders.
    
    Parameters:
        df: DataFrame with columns 'task', 'order', 'subject', 'model', 'value'
        model_a: First model name
        model_b: Second model name
    
    Returns:
        pd.DataFrame: A DataFrame with task, order, p-value, and effect size (mean diff)
    """
    results = []

    grouped = df.groupby(['task', 'order'])
    for (task, order), group in grouped:
        # Pivot to get one row per subject, one column per model
        pivoted = group[group['model'].isin([model_a, model_b])].pivot(index='subject', columns='model', values='value')

        # Drop any rows with NaNs (subjects missing from either model)
        pivoted = pivoted.dropna()

        if pivoted.shape[0] < 2:
            continue  # skip if not enough subjects

        a_values = pivoted[model_a].values
        b_values = pivoted[model_b].values

        # Paired permutation test using mean difference as statistic
        result = permutation_test((a_values, b_values),
                                  statistic=lambda x, y: (x - y).mean(),
                                  permutation_type='samples',
                                  vectorized=False,
                                  alternative='less',
                                  n_resamples=10000,
                                  random_state=SEED)

        results.append({
            'task': task,
            'order': order,
            'model_a': model_a,
            'model_b': model_b,
            'mean_diff': (a_values - b_values).mean(),
            'p_value': result.pvalue,
            'n_subjects': len(pivoted)
        })

    return pd.DataFrame(results)

In [3]:
def bootstrap_ci_mean(data, n_bootstrap=1000, ci_level=0.95):
    n = len(data)
    bootstrap_samples = np.random.choice(data, size=(n_bootstrap, n), replace=True)
    means = np.mean(bootstrap_samples, axis=1)
    lower_ci = np.percentile(means, (1 - ci_level) / 2 * 100)
    upper_ci = np.percentile(means, (1 + ci_level) / 2 * 100)
    return lower_ci, upper_ci

In [4]:
def load_all_accuracies(base_dir):
    """
    Load accuracy CSVs organized by model and combine into a single DataFrame.
    
    Returns:
        pd.DataFrame with columns ['subject', 'task', 'model', 'value', 'order']
    """
    all_dfs = []
    
    # Define models and their corresponding directory names and file patterns
    models_config = {
        'pt': (f'{base_dir}/pretrained', 'pt'),
        'pt_calib': (f'{base_dir}/deepmreye_and_calib', 'pt_calib'),
        'pt_gaze': (f'{base_dir}/deepmreye_and_eyestate_tracking', 'pt_gaze')
    }
    
    for model_name, (model_dir, file_suffix) in models_config.items():
        # Load all 8 CSV files for this model (4 tasks × 2 orders)
        for task_num in range(1, 5):
            for order_type in ['ordered', 'shuffled']:
                filename = f'{model_dir}/accuracies_task_{task_num}_{order_type}_{file_suffix}.csv'
                
                try:
                    df = pd.read_csv(filename, index_col=0)  # First column is index (subject)
                    accuracy_values = df.iloc[:, 0].values  # Get the accuracy column
                    
                    # Create reshaped dataframe
                    df_reshaped = pd.DataFrame({
                        'subject': [f'sub-{i+1:02d}' for i in range(len(accuracy_values))],
                        'task': f'Task {task_num} Accuracy',
                        'model': model_name,
                        'value': accuracy_values,
                        'order': order_type
                    })
                    all_dfs.append(df_reshaped)
                    print(f'✓ Loaded {filename}')
                except FileNotFoundError:
                    print(f'✗ File not found: {filename}')
    
    # Combine all dataframes
    df_all_accuracies = pd.concat(all_dfs, ignore_index=True)
    
    return df_all_accuracies


# Load the data - update the path to your Desktop or where you saved the CSV files
df_all_accuracies = load_all_accuracies(base_dir='/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred/')

print("\n" + "="*50)
print("Data loaded successfully!")
print("="*50)
print(f"Shape: {df_all_accuracies.shape}")
print(f"\nColumns: {list(df_all_accuracies.columns)}")
print(f"\nModels: {df_all_accuracies['model'].unique()}")
print(f"Tasks: {df_all_accuracies['task'].unique()}")
print(f"Orders: {df_all_accuracies['order'].unique()}")
print(f"\nFirst few rows:")
print(df_all_accuracies.head(10))

✓ Loaded /Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred//pretrained/accuracies_task_1_ordered_pt.csv
✓ Loaded /Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred//pretrained/accuracies_task_1_shuffled_pt.csv
✓ Loaded /Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred//pretrained/accuracies_task_2_ordered_pt.csv
✓ Loaded /Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred//pretrained/accuracies_task_2_shuffled_pt.csv
✓ Loaded /Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred//pretrained/accuracies_task_3_ordered_pt.csv
✓ Loaded /Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/classifier/pred//pretrained/accuracies_task_3

In [5]:
# Run paired permutation tests
sig_df_pt_vs_calib = paired_permutation_tests(df_all_accuracies, model_a='pt', model_b='pt_calib')
sig_df_pt_vs_gaze = paired_permutation_tests(df_all_accuracies, model_a='pt', model_b='pt_gaze')

# Combine results
sig_df = pd.concat([sig_df_pt_vs_calib, sig_df_pt_vs_gaze], ignore_index=True)

print("\n=== STATISTICAL TEST RESULTS ===")
print(sig_df.to_string())


=== STATISTICAL TEST RESULTS ===
               task     order model_a   model_b  mean_diff   p_value  n_subjects
0   Task 1 Accuracy   ordered      pt  pt_calib  -0.052778  0.501250          15
1   Task 1 Accuracy  shuffled      pt  pt_calib  -0.011111  0.390661          15
2   Task 2 Accuracy   ordered      pt  pt_calib  -0.041667  0.501250          15
3   Task 2 Accuracy  shuffled      pt  pt_calib  -0.100000  0.002100          15
4   Task 3 Accuracy   ordered      pt  pt_calib  -0.008333  0.484052          15
5   Task 3 Accuracy  shuffled      pt  pt_calib  -0.002778  0.497450          15
6   Task 4 Accuracy   ordered      pt  pt_calib  -0.002778  0.498650          15
7   Task 4 Accuracy  shuffled      pt  pt_calib  -0.033333  0.128487          15
8   Task 1 Accuracy   ordered      pt   pt_gaze  -0.052778  0.501250          15
9   Task 1 Accuracy  shuffled      pt   pt_gaze  -0.130556  0.000100          15
10  Task 2 Accuracy   ordered      pt   pt_gaze  -0.041667  0.501250       

In [6]:
# Summary of significant results (p < 0.05)
sig_results = sig_df[sig_df['p_value'] < 0.05]
print("\n=== SIGNIFICANT RESULTS (p < 0.05) ===")
if len(sig_results) > 0:
    for _, row in sig_results.iterrows():
        print(f"  {row['task']} ({row['order']})")
        print(f"    {row['model_a']} vs {row['model_b']}: p = {row['p_value']:.6f}, mean_diff = {row['mean_diff']:.4f}")
else:
    print("  No significant differences found (p < 0.05)")


=== SIGNIFICANT RESULTS (p < 0.05) ===
  Task 2 Accuracy (shuffled)
    pt vs pt_calib: p = 0.002100, mean_diff = -0.1000
  Task 1 Accuracy (shuffled)
    pt vs pt_gaze: p = 0.000100, mean_diff = -0.1306
  Task 2 Accuracy (shuffled)
    pt vs pt_gaze: p = 0.000200, mean_diff = -0.1389
  Task 4 Accuracy (shuffled)
    pt vs pt_gaze: p = 0.026197, mean_diff = -0.0694


In [7]:
# Summary statistics with 95% confidence intervals
print("\n=== ACCURACY SUMMARY (with 95% Confidence Intervals) ===")

def calculate_ci_95_bootstrap(group):
    ci_lower, ci_upper = bootstrap_ci_mean(group.values)
    return pd.Series({
        'mean': group.mean(),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'std': group.std(),
        'count': len(group)
    })

summary = df_all_accuracies.groupby(['task', 'order', 'model'])['value'].apply(calculate_ci_95_bootstrap).unstack()

display_summary = pd.DataFrame({
    'Mean (%)': summary['mean'] * 100,
    'CI Lower (%)': summary['ci_lower'] * 100,
    'CI Upper (%)': summary['ci_upper'] * 100,
    'Std (%)': summary['std'] * 100,
    'N': summary['count'].astype(int)
})
print(display_summary.round(2).to_string())


=== ACCURACY SUMMARY (with 95% Confidence Intervals) ===
                                   Mean (%)  CI Lower (%)  CI Upper (%)  Std (%)   N
task            order    model                                                      
Task 1 Accuracy ordered  pt           94.72         84.17        100.00    20.44  15
                         pt_calib    100.00        100.00        100.00     0.00  15
                         pt_gaze     100.00        100.00        100.00     0.00  15
                shuffled pt           86.67         79.17         92.50    13.93  15
                         pt_calib     87.78         80.00         93.61    13.77  15
                         pt_gaze      99.72         99.17        100.00     1.08  15
Task 2 Accuracy ordered  pt           95.83         87.50        100.00    16.14  15
                         pt_calib    100.00        100.00        100.00     0.00  15
                         pt_gaze     100.00        100.00        100.00     0.00  15
       